Before creating the Spark Session, environment variables were configured using `PYSPARK_SUBMIT_ARGS` to pass Java options to Spark. These settings allow the Spark Driver and Executors to work correctly with newer Java versions by enabling compatibility with the deprecated Java Security Manager. This configuration helps prevent startup and runtime errors that may occur when running PySpark on modern Java environments and ensures that Spark initializes successfully.

In [1]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf "spark.driver.extraJavaOptions=-Djava.security.manager=allow" '
    '--conf "spark.executor.extraJavaOptions=-Djava.security.manager=allow" '
    'pyspark-shell'
)

# Step 1: Create Spark Session

A Spark Session is the entry point for working with Apache Spark. It provides access to Spark functionalities such as DataFrames, SQL operations, and distributed data processing. In this step, a Spark Session named **"Spark Assignment"** is created using the builder pattern. The `getOrCreate()` method initializes a new Spark Session if one does not already exist, otherwise it returns the existing session. This session will be used throughout the assignment to perform data loading, transformation, filtering, aggregation, and analysis tasks.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Assignment") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


# Step 2: Load Dataset
The Superstore dataset is loaded into a Spark DataFrame using Spark's CSV reader. Several options are configured to ensure the data is read correctly. The `header=true` option treats the first row as column names, while `inferSchema=true` automatically detects appropriate data types for each column. The `multiLine=true` option allows Spark to handle records that span multiple lines, and the `escape` option manages special characters and quotation marks within text fields. After loading the dataset, the first five rows are displayed to verify that the data has been imported successfully.

In [22]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv("C:/Users/iamsu/OneDrive/Desktop/Assignment 05/data/Sample - Superstore.csv")
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Display Dataset Schema

In [4]:
df.printSchema()


root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



# Step 3: Explore Data

In [5]:
print(df.columns)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


# Step 4:Data Cleaning

In [6]:
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [7]:
print("Rows before removing duplicates:")
print(df.count())

Rows before removing duplicates:
9994


In [8]:
df = df.dropDuplicates()

print("Rows after removing duplicates:")
print(df.count())

Rows after removing duplicates:
9994


In [9]:
df = df.na.drop()

In [10]:
df.describe().show()

+-------+-----------------+--------------+----------+---------+--------------+-----------+------------------+-----------+-------------+--------+-------+-----------------+-------+---------------+----------+------------+--------------------+-----------------+------------------+------------------+------------------+
|summary|           Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|    City|  State|      Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|            Sales|          Quantity|          Discount|            Profit|
+-------+-----------------+--------------+----------+---------+--------------+-----------+------------------+-----------+-------------+--------+-------+-----------------+-------+---------------+----------+------------+--------------------+-----------------+------------------+------------------+------------------+
|  count|             9994|          9994|      9994|  

# Step 5: Filter Data

In [11]:
df.filter(df.Region == "West").show()

+------+--------------+----------+----------+--------------+-----------+--------------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|       Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+--------------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+---------+
|  1142|CA-2014-146969| 9/29/2014| 10/3/2014|Standard Class|   AP-10915|      Arthur Prichep|   Consumer|United States|  Los Angeles|California|      90045|  West|FUR-FU-10004188|      Furniture| Furnis

In [12]:
df.filter(df.Category == "Furniture").show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------+------------+--------------------+--------------------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|         State|Postal Code| Region|     Product ID| Category|Sub-Category|        Product Name|               Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------+------------+--------------------+--------------------+--------+--------+---------+
|  1142|CA-2014-146969| 9/29/2014| 10/3/2014|Standard Class|   AP-10915|   Arthur Prichep|   Consumer|United States|  Los Angeles|    California|      90045|   West|FUR-FU-100

In [23]:
df.filter(df.Sales > 500).show()

+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|    Segment|      Country|           City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|     2|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute|   Consumer|United States|      Henderson|    Kentucky|      42420|  South|FUR-CH-10000454|      Furniture| 

# Step 6: Transform Data

In [24]:
from pyspark.sql.functions import round

df = df.withColumn(
    "Profit_Margin",
    round((col("Profit")/col("Sales"))*100,2)
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|Profit_Margin|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798| 

# Step 7: Aggregation

In [25]:
print("Total Rows =", df.count())

Total Rows = 9994


In [26]:
from pyspark.sql.functions import avg

df.select(
    avg("Sales").alias("Average Sales")
).show()

+-----------------+
|    Average Sales|
+-----------------+
|229.8580008304938|
+-----------------+



In [27]:
from pyspark.sql.functions import min,max

df.select(
    min("Sales").alias("Min Sales"),
    max("Sales").alias("Max Sales")
).show()

+---------+---------+
|Min Sales|Max Sales|
+---------+---------+
|    0.444| 22638.48|
+---------+---------+



# Step 8: Group Data

In [28]:
df.groupBy("Region").count().show()

+-------+-----+
| Region|count|
+-------+-----+
|  South| 1620|
|Central| 2323|
|   East| 2848|
|   West| 3203|
+-------+-----+



In [29]:
from pyspark.sql.functions import sum

df.groupBy("Category") \
  .agg(sum("Sales").alias("Total Sales")) \
  .show()

+---------------+-----------------+
|       Category|      Total Sales|
+---------------+-----------------+
|Office Supplies|719047.0320000029|
|      Furniture|741999.7952999998|
|     Technology|836154.0329999966|
+---------------+-----------------+



In [30]:
from pyspark.sql.functions import avg

df.groupBy("Region") \
  .agg(avg("Profit").alias("Average Profit")) \
  .show()

+-------+------------------+
| Region|    Average Profit|
+-------+------------------+
|  South|28.857673024691398|
|Central| 17.09270878174773|
|   East|32.135807584269756|
|   West|33.849031813924505|
+-------+------------------+



## Step 9: Advanced Concept

1. Wide Transformations

Wide transformations are Spark operations that require data to move across different partitions before producing the result.

Examples: groupBy()
          join()
          distinct()
          reduceByKey()



2. Shuffle

Shuffle is the process of redistributing data across partitions during execution.

It occurs during operations such as:

groupBy()
join()
distinct()

Example: df.groupBy("Region").count()


# Step 10: Build Simple Pipeline

In [42]:
from pyspark.sql.functions import count, avg, col


pipeline_df = spark.read.csv(
    r"C:\Users\iamsu\OneDrive\Desktop\Assignment 05\data\Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully")
print("Total Rows:", pipeline_df.count())

before_rows = pipeline_df.count()

pipeline_df = pipeline_df.dropDuplicates()

after_rows = pipeline_df.count()

print("Duplicates Removed:", before_rows - after_rows)

west_df = pipeline_df.filter(
    pipeline_df.Region == "West"
)

print("West Region Records:")
print(west_df.count())


west_df = west_df.withColumnRenamed(
    "Sub-Category",
    "SubCategory"
)


west_df = west_df.withColumn(
    "High_Value_Order",
    col("Sales") > 500
)

print("Transformation Applied")

from pyspark.sql.functions import count

result = west_df.groupBy("Category") \
    .agg(
        count("*").alias("Total_Orders")
    )


print("Final Aggregated Results")

result.show(truncate=False)

Dataset Loaded Successfully
Total Rows: 9994
Duplicates Removed: 0
West Region Records:
3203
Transformation Applied
Final Aggregated Results
+---------------+------------+
|Category       |Total_Orders|
+---------------+------------+
|Office Supplies|1897        |
|Furniture      |707         |
|Technology     |599         |
+---------------+------------+



In [59]:
pdf = df.toPandas()

pdf.to_csv("C:\\Users\\iamsu\\OneDrive\\Desktop\\Assignment 05\\output\\result.csv", index=False)

print("Done")

Done
